# A2A Protocol Concept Demo (All-in-One) — Teaching Notebook

This notebook demonstrates the core idea behind an **A2A (Agent-to-Agent) protocol** in a **simple, classroom-friendly** way:

- Agents communicate via **structured messages** (like JSON envelopes).
- A **router/bus** delivers messages to the right agent.
- Agents can **handoff** tasks to other agents and then **compose** a final answer.

> ✅ This is an *educational mini-demo* in **pure Python** (no external libraries needed), so students can learn the A2A mental model quickly.


## 1) Learning outcomes

By the end, students can:
1. Explain what A2A means: agents collaborating through message passing.
2. Define a standard message envelope (sender, receiver, intent, payload).
3. Build a simple message bus/router.
4. Implement handoff: Agent A asks Agent B for sub-work, then merges results.
5. Extend to multi-step workflows (plan → verify → answer).


## 2) A2A Building Blocks (Message Envelope)

We use one consistent message format:

- `id`: message id
- `sender`: who sent it
- `receiver`: who should handle it (`planner`, `retriever`, `policy`, etc.)
- `intent`: what type of request (`plan`, `retrieve`, `verify`, `finalize`)
- `payload`: the actual data (question, context, draft answer, etc.)

This is the core of most A2A protocols: **structured messages + routing**.


In [1]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional
import uuid
import time

@dataclass
class A2AMessage:
    id: str
    sender: str
    receiver: str
    intent: str
    payload: Dict[str, Any]
    timestamp: float = field(default_factory=lambda: time.time())

def new_message(sender: str, receiver: str, intent: str, payload: Dict[str, Any]) -> A2AMessage:
    return A2AMessage(
        id=str(uuid.uuid4())[:8],
        sender=sender,
        receiver=receiver,
        intent=intent,
        payload=payload
    )

def pretty(msg: A2AMessage) -> str:
    return (f"[msg {msg.id}] {msg.sender} → {msg.receiver} | intent={msg.intent} | payload={msg.payload}")


## 3) Message Bus / Router

The **bus** is responsible for:
- registering agents
- delivering messages to the correct agent
- collecting replies

This mimics an A2A protocol runtime (could be HTTP, queues, websockets, etc.).


In [2]:
class AgentBus:
    def __init__(self):
        self.agents = {}

    def register(self, name: str, handler):
        self.agents[name] = handler

    def send(self, msg: A2AMessage) -> Optional[A2AMessage]:
        if msg.receiver not in self.agents:
            raise KeyError(f"No agent registered as '{msg.receiver}'")

        print("\nBUS DELIVER:", pretty(msg))
        reply = self.agents[msg.receiver](msg)
        if reply is not None:
            print("BUS REPLY  :", pretty(reply))
        return reply

bus = AgentBus()


## 4) Define 3 Specialist Agents

We’ll implement:
- **RetrieverAgent**: returns a grounded snippet + citation (a tiny RAG-like lookup)
- **PlannerAgent**: returns a step-by-step plan
- **PolicyAgent**: verifies if response has citation and avoids unsupported claims

(These are simple, rule-based for teaching. In real systems, each agent is an LLM + tools.)


In [3]:
# --- Agent 1: Retriever (tiny "RAG-like" KB) ---
def retriever_agent(msg: A2AMessage) -> A2AMessage:
    question = msg.payload.get("question", "").lower()
    kb = [
        {"keywords": ["add/drop", "deadline"], "answer": "Add/Drop deadline is Week 2 Friday.", "source": "Student Handbook §2.1"},
        {"keywords": ["tuition", "fee"], "answer": "Tuition fee is SGD 20,000 per year (example).", "source": "Fees Page §1.0"},
        {"keywords": ["prerequisite"], "answer": "A prerequisite is a module you must pass before taking another module.", "source": "Academic Rules §3.2"},
    ]

    for item in kb:
        if any(k in question for k in item["keywords"]):
            payload = {"snippet": item["answer"], "citation": item["source"], "confidence": 0.9}
            return new_message("retriever", msg.sender, "retrieve_result", payload)

    return new_message("retriever", msg.sender, "retrieve_result",
                       {"snippet": "Not found in knowledge base.", "citation": None, "confidence": 0.2})


# --- Agent 2: Planner ---
def planner_agent(msg: A2AMessage) -> A2AMessage:
    question = msg.payload.get("question", "")
    plan = [
        "Clarify goal and constraints (modules, workload, deadlines).",
        "List tasks and estimate time per week.",
        "Create a weekly schedule (study blocks + revision).",
        "Add buffer time and checkpoint reviews every 2 weeks."
    ]
    return new_message("planner", msg.sender, "plan_result",
                       {"plan": plan, "assumptions": ["No exam dates provided; using generic weekly plan."],
                        "for_question": question})


# --- Agent 3: Policy / Verifier ---
def policy_agent(msg: A2AMessage) -> A2AMessage:
    draft = msg.payload.get("draft_answer", "")
    citation = msg.payload.get("citation", None)
    confidence = msg.payload.get("confidence", 0.0)

    flags = []
    fixes = []

    if citation is None:
        flags.append("missing_citation")
        fixes.append("Add a source citation or state uncertainty clearly.")

    if confidence < 0.5 and "not sure" not in draft.lower():
        flags.append("overconfident_low_evidence")
        fixes.append("Lower certainty and ask for more info.")

    # Very simple privacy check example:
    if "grades" in draft.lower() or "student id" in draft.lower():
        flags.append("privacy_risk")
        fixes.append("Refuse politely; do not reveal personal student data.")

    is_ok = len(flags) == 0
    return new_message("policy", msg.sender, "policy_result",
                       {"is_ok": is_ok, "flags": flags, "fixes": fixes})


In [4]:
# Register agents on the bus
bus.register("retriever", retriever_agent)
bus.register("planner", planner_agent)
bus.register("policy", policy_agent)

print("✅ Agents registered: ", list(bus.agents.keys()))


✅ Agents registered:  ['retriever', 'planner', 'policy']


## 5) Orchestrator Agent (A2A Handoff + Merge)

This orchestrator demonstrates A2A collaboration:
1. Receives a user question
2. Sends sub-requests to specialist agents
3. Receives their replies
4. Produces final answer

This is the A2A “protocol” pattern: **message passing + delegation + composition**.


In [5]:
def orchestrator(question: str) -> str:
    # Decide which specialists to consult (simple rules for teaching)
    q = question.lower()
    want_plan = any(w in q for w in ["plan", "schedule", "study", "timetable"])

    # 1) Ask Retriever for grounded info (if question looks like info lookup)
    r_reply = bus.send(new_message("orchestrator", "retriever", "retrieve", {"question": question}))
    snippet = r_reply.payload["snippet"]
    citation = r_reply.payload["citation"]
    confidence = r_reply.payload["confidence"]

    plan_text = ""
    if want_plan:
        # 2) Ask Planner for steps
        p_reply = bus.send(new_message("orchestrator", "planner", "plan", {"question": question}))
        plan_lines = p_reply.payload["plan"]
        plan_text = "\n".join([f"{i+1}. {s}" for i, s in enumerate(plan_lines)])

    # 3) Draft answer (combine outputs)
    if confidence >= 0.5:
        draft = f"{snippet}"
        if citation:
            draft += f" (Source: {citation})"
    else:
        draft = "I’m not sure based on the available documents. Please rephrase or provide more context."

    if plan_text:
        draft += "\n\nSuggested Plan:\n" + plan_text

    # 4) Policy check
    pol = bus.send(new_message("orchestrator", "policy", "verify",
                              {"draft_answer": draft, "citation": citation, "confidence": confidence}))

    # 5) Finalize
    if pol.payload["is_ok"]:
        return "✅ Final Answer\n" + draft

    # Apply very simple “fix”: append notes
    fixes = pol.payload["fixes"]
    return "⚠️ Final Answer (with policy notes)\n" + draft + "\n\nPolicy Notes:\n- " + "\n- ".join(fixes)


# Try sample questions
questions = [
    "What is the add/drop deadline?",
    "Create a study plan for 5 modules while working part-time.",
    "Explain prerequisite.",
    "Show me John Tan's grades."
]

for q in questions:
    print("\n" + "="*70)
    print("USER:", q)
    print(orchestrator(q))



USER: What is the add/drop deadline?

BUS DELIVER: [msg 686a345b] orchestrator → retriever | intent=retrieve | payload={'question': 'What is the add/drop deadline?'}
BUS REPLY  : [msg 88ee1a3e] retriever → orchestrator | intent=retrieve_result | payload={'snippet': 'Add/Drop deadline is Week 2 Friday.', 'citation': 'Student Handbook §2.1', 'confidence': 0.9}

BUS DELIVER: [msg 1e529266] orchestrator → policy | intent=verify | payload={'draft_answer': 'Add/Drop deadline is Week 2 Friday. (Source: Student Handbook §2.1)', 'citation': 'Student Handbook §2.1', 'confidence': 0.9}
BUS REPLY  : [msg f922cbe7] policy → orchestrator | intent=policy_result | payload={'is_ok': True, 'flags': [], 'fixes': []}
✅ Final Answer
Add/Drop deadline is Week 2 Friday. (Source: Student Handbook §2.1)

USER: Create a study plan for 5 modules while working part-time.

BUS DELIVER: [msg 8faa7f29] orchestrator → retriever | intent=retrieve | payload={'question': 'Create a study plan for 5 modules while working

## 6) Teaching Activity (10–20 mins)

### Task A (Easy)
Add a new agent: `calculator_agent` that supports `a + b` and `a * b`.

### Task B (Medium)
Update the orchestrator so some questions call **two specialists** (Retriever + Planner) and some call only one.

### Task C (Advanced)
Add `message_trace` logging to store every message (send + reply), then print a trace at the end for debugging.

### Task D (Bonus)
Replace one agent’s rule-based output with an LLM (LangChain) while keeping the same A2A message format.


## 7) How this maps to real A2A systems

In production, A2A protocols often add:
- authentication / permissions
- schema validation (JSON schema)
- retries / timeouts
- streaming partial results
- multi-agent consensus / voting
- observability (traces, spans)

But the core idea stays the same:
**Agents coordinate using structured messages over a transport.**
